# Feature Engineering & Pipeline Transformation

## Business Objectives (8 Key Questions for Dashboard Visualizations):
1. **Sales & Profit Trends:** How do sales and profits perform over time (Yearly/Monthly)?
2. **Regional Profitability:** Which geographical regions drive the highest profit margins?
3. **Category Breakdown:** What are the top-performing categories and sub-categories by profit?
4. **Discount Impact:** How do varying discount levels directly affect profit margins?
5. **Logistics & SLA:** What is the average shipping duration across different ship modes, and where do delays occur?
6. **VIP Customer Identification:** Who are the Top 10 customers based on Lifetime Spend (LTV)?
7. **Order Size Distribution:** How are sales distributed across order volume tiers (Small, Medium, Bulk)?
8. **Weekly Purchasing Patterns:** Which days of the week experience the highest order volume?

In [1]:
import pandas as pd
import numpy as np
import pyarrow 

In [2]:
df = pd.read_pickle("../data/processed/superstore_cleaned.pkl")

In [3]:
df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country/Region,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2018-152156,2018-11-08,2018-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.959991,2,0.00,41.913601
1,2,CA-2018-152156,2018-11-08,2018-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.940002,3,0.00,219.582001
2,3,CA-2018-138688,2018-06-12,2018-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.620000,2,0.00,6.871400
3,4,US-2017-108966,2017-10-11,2017-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.577515,5,0.45,-383.031006
4,5,US-2017-108966,2017-10-11,2017-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.368000,2,0.20,2.516400


In [4]:
class FeatureEngineer:
    def __init__(self, df):
        self.df = df.copy()

In [5]:
def add_time_features(self):
        """
        1. Date & Logistics Features (Answers Questions 1, 5, 8)
        - Shipping Duration & Delay Status
        - Time Breakdown (Year, Month, Day, Quarter, Weekend)
        """
        
        self.df['Shipping_Duration'] = (self.df['Ship Date'] - self.df['Order Date']).dt.days

        self.df['Is_Delayed'] = (self.df['Shipping_Duration'] > 4).astype(int)

        self.df['Order_Year'] = self.df['Order Date'].dt.year
        self.df['Order_Month'] = self.df['Order Date'].dt.month
        self.df['Order_Month_Name'] = self.df['Order Date'].dt.month_name()
        self.df['Order_Day_Name'] = self.df['Order Date'].dt.day_name()
        self.df['Order_Quarter'] = self.df['Order Date'].dt.to_period('Q').astype(str)
        self.df['Is_Weekend'] = self.df['Order Date'].dt.dayofweek.isin([5, 6]).astype(int)

        return self.df

FeatureEngineer.add_time_features = add_time_features

In [6]:
def add_financial_features(self):
        """
        2. Financial & Profitability Features (Answers Questions 2, 3, 4)
        - Profit Margin & Unit Price
        - Absolute Discount Amount
        """
        
        self.df['Profit_Margin'] = np.where(
            self.df['Sales'] != 0,
            self.df['Profit'] / self.df['Sales'],
            0
        )

        self.df['Discount_Amount'] = self.df['Sales'] * self.df['Discount']

        self.df['Unit_Price'] = np.where(
            self.df['Quantity'] != 0,
            self.df['Sales'] / self.df['Quantity'],
            0
        )
        return self.df

FeatureEngineer.add_financial_features = add_financial_features

In [7]:
def add_customer_and_order_features(self):
        """
        3. Customer & Order Segmentation Features (Answers Questions 6, 7)
        - Customer Lifetime Spend & Order Counts
        - Order Size Categorization (Small, Medium, Bulk)
        """
        customer_counts = self.df.groupby('Customer ID', observed=False)['Order ID'].transform('nunique')
        self.df['Customer_Order_Count'] = customer_counts

        customer_spend = self.df.groupby('Customer ID', observed=False)['Sales'].transform('sum')
        self.df['Customer_Total_Spend'] = customer_spend


        self.df['Order_Size'] = pd.cut(
            self.df['Quantity'],
            bins=[0, 2, 5, np.inf],
            labels=['Small', 'Medium', 'Bulk'],
            right=True
        )

        return self.df
FeatureEngineer.add_customer_and_order_features = add_customer_and_order_features

In [8]:
def transform_all(self) -> pd.DataFrame:
        """Executes all feature engineering steps in sequence."""
        self.add_time_features()
        self.add_financial_features()
        self.add_customer_and_order_features()
        return self.df
FeatureEngineer.transform_all = transform_all

In [9]:
fe = FeatureEngineer(df)
df_featured = fe.transform_all()

engineered_cols = [
    'Shipping_Duration', 'Is_Delayed', 'Profit_Margin',
    'Discount_Amount', 'Unit_Price', 'Customer_Total_Spend',
    'Customer_Order_Count', 'Order_Size'
]


print(f"\nTotal Columns: {df_featured.shape[1]}")

df_featured[engineered_cols].head()


Total Columns: 35


,Shipping_Duration,Is_Delayed,Profit_Margin,Discount_Amount,Unit_Price,Customer_Total_Spend,Customer_Order_Count,Order_Size
0,3,0,0.1600,0.000000,130.979996,1148.780029,3,Small
1,3,0,0.3000,0.000000,243.980001,1148.780029,3,Medium
2,4,0,0.4700,0.000000,7.310000,1119.483032,5,Small
3,7,1,-0.4000,430.909882,191.515503,2602.575439,6,Medium
4,7,1,0.1125,4.473600,11.184000,2602.575439,6,Small


In [10]:
output_path = "../data/processed/superstore_features.pkl"
df_featured.to_pickle(output_path)

df_featured.to_csv("../data/processed/superstore_features.csv", index=False)

